# Lab 7 · GF-CF

The runnable companion of the lab page, `docs/labs/07-gfcf.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "gfcf"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
print("non-zero weights in the linear filter:", model.linear.nnz)
print(model.fit_info)

**Exercise.** How many non-zero weights does each item have in the linear filter, on average and at most? Compare with ItemKNN's `knn_neighbors`.

<details><summary>Hint</summary>

Row lengths of a CSR matrix: `np.diff(model.linear.indptr)`.

</details>

<details><summary>Solution</summary>

```python
import numpy as np

per_item = np.diff(model.linear.indptr)
print(per_item.mean(), per_item.max())
```
Popular items are linked to almost every other item: GF-CF keeps every link, where ItemKNN keeps the top k.

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "gfcf_alpha", [0.0, 0.3, 0.6, 1.0], start="best")
lab.plot_sweep(table)
table

**Your prediction for `gfcf_alpha`, and what you saw:** ...

In [ ]:
table = lab.sweep(METHOD, DATASET, "gfcf_k", [64, 256, 1024], start="best")
lab.plot_sweep(table)
table

**Your prediction for `gfcf_k`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh gfcf:linear-only
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `linear-only` is ready to run; `pruned`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_gfcf_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:linear-only")

In [ ]:
lab.compare(METHOD, f"{METHOD}:pruned")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`smooth` in `labs/07-gfcf/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:smooth")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:smooth", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.